# Ótimos de MDS, MVC e MIS com Gurobi

Execute as células de preparação até **Formular e resolver as PLIs**. Depois, escolha a seção de execução individual, parada por gap percentual ou lote.

| Problema | Objetivo | Restrições |
| --- | --- | --- |
| MDS | $\min \sum_v x_v$ | $x_v + \sum_{u\in N(v)} x_u \ge 1$ para cada vértice $v$ |
| MVC | $\min \sum_v x_v$ | $x_u+x_v \ge 1$ para cada aresta $\{u,v\}$ |
| MIS | $\max \sum_v x_v$ | $x_u+x_v \le 1$ para cada aresta $\{u,v\}$ |

Cada $x_v$ é binário. `otimo` é preenchido quando o Gurobi retorna `OPTIMAL` e o limite `ObjBound` coincide com o valor inteiro da solução. Isso segue o critério de otimalidade do Gurobi, sujeito às tolerâncias numéricas do solver.

Abra o notebook na raiz do projeto ou em `notebooks/`. Se necessário, instale as dependências no kernel com `%pip install gurobipy==13.0.3 networkx pandas`. O Gurobi usa a licença configurada no ambiente.

[Status do Gurobi](https://docs.gurobi.com/projects/optimizer/en/13.0/reference/numericcodes/statuscodes.html) · [Atributos do modelo](https://docs.gurobi.com/projects/optimizer/en/13.0/reference/attributes/model.html#objbound)

In [31]:
import csv
import gzip
import gc
import re
from math import isfinite
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent))

import gurobipy as gp
import networkx as nx
import pandas as pd
from gurobipy import GRB
from IPython.display import display

from src.complex_networks.config import PROJECT_ROOT, RAW_DATA_DIR, RESULTS_DIR


MOSTRAR_LOG = True
PROBLEMAS_SUPORTADOS = ("MDS", "MVC", "MIS")

# Configuração do lote
PROBLEMAS = ("MDS",)

CSV_RESULTADOS = RESULTS_DIR / "otimos_gap1_gurobi.csv"


print("Gurobi", gp.gurobi.version())

Gurobi (14, 0, 0)


## Ler os grafos

Os arquivos `.txt` e `.txt.gz` são listas de arestas com vértices inteiros. A leitura trata as arestas como não direcionadas, agrega duplicatas e remove laços mantendo seus vértices. Um isolado também pode ser declarado com um único rótulo na linha.

In [32]:
def ler_grafo(arquivo):
    G = nx.Graph()
    n_declarado = None
    abrir = gzip.open if arquivo.suffix == ".gz" else open

    with abrir(arquivo, "rt", encoding="utf-8-sig") as entrada:
        for numero, linha in enumerate(entrada, start=1):
            cabecalho = re.match(r"^\s*#\s*Nodes:\s*(\d+)\b", linha, re.IGNORECASE)
            if cabecalho:
                n_declarado = int(cabecalho[1])

            campos = linha.split("#", 1)[0].split()
            if not campos:
                continue
            try:
                u = int(campos[0])
                v = int(campos[1]) if len(campos) > 1 else u
            except ValueError as erro:
                raise ValueError(f"Linha {numero} com rótulo de vértice inválido.") from erro
            if u == v:
                G.add_node(u)
            else:
                G.add_edge(u, v)

    if n_declarado is not None and len(G) != n_declarado:
        raise ValueError(f"Cabeçalho com {n_declarado} vértices e {len(G)} presentes no arquivo.")
    return G

## Formular e resolver as PLIs

In [33]:
STATUS_GUROBI = {
    getattr(GRB.Status, nome): nome for nome in dir(GRB.Status) if nome.isupper()
}


def resolver_pli(G, problema, env, *, gap_max_pct=0):
    if problema not in PROBLEMAS_SUPORTADOS:
        raise ValueError("Escolha um dos problemas MDS, MVC ou MIS.")
    if type(gap_max_pct) not in (int, float) or not isfinite(gap_max_pct) or gap_max_pct < 0:
        raise ValueError("Informe um gap percentual finito e não negativo. Use 5 para 5%.")

    with gp.Model(problema, env=env) as model:
        x = model.addVars(G.nodes(), vtype=GRB.BINARY, name="x")

        sentido = GRB.MAXIMIZE if problema == "MIS" else GRB.MINIMIZE
        model.setObjective(gp.quicksum(x[v] for v in G.nodes()), sentido)

        if problema == "MDS":
            for v in G.nodes():
                model.addConstr(x[v] + gp.quicksum(x[u] for u in G.neighbors(v)) >= 1)
        elif problema == "MVC":
            for u, v in G.edges():
                model.addConstr(x[u] + x[v] >= 1)
        else:  # MIS
            for u, v in G.edges():
                model.addConstr(x[u] + x[v] <= 1)

        # Em minimização, o Gurobi divide pelo LS. A conversão expressa a tolerância sobre o LI.
        model.Params.MIPGap = (
            gap_max_pct / (100 + gap_max_pct)
            if sentido == GRB.MINIMIZE
            else gap_max_pct / 100
        )
        model.Params.MIPGapAbs = 0
        model.optimize()

        status = model.Status
        melhor = round(model.ObjVal) if model.SolCount else None
        limite = getattr(model, "ObjBound", None)
        limite = limite if limite is not None and abs(limite) < GRB.INFINITY else None
        otimo = melhor if status == GRB.OPTIMAL and melhor is not None and melhor == limite else None

        return {
            "status_gurobi": STATUS_GUROBI.get(status, str(status)),
            "otimo": otimo,
            "melhor_viavel": melhor,
            "limite_gurobi": limite,
            "tempo_ate_gap_s": round(model.Runtime, 6) if status == GRB.OPTIMAL else None,
        }

In [34]:
def executar_instancia(grafo, problema, *, gap_max_pct=0):
    problema = problema.strip().upper()
    caminho = Path(grafo).expanduser()
    candidatos = [caminho]
    if caminho.parent == Path("."):
        if caminho.name.endswith((".txt", ".txt.gz")):
            candidatos.append(RAW_DATA_DIR / caminho)
        else:
            candidatos.extend(
                RAW_DATA_DIR / f"{caminho.name}{extensao}"
                for extensao in (".txt.gz", ".txt")
            )

    arquivo = next((p for p in candidatos if p.is_file()), None)
    if arquivo is None:
        raise FileNotFoundError(
            f"Grafo {grafo!r} não encontrado. Informe o nome em {RAW_DATA_DIR} "
            "ou o caminho completo do arquivo."
        )

    G = ler_grafo(arquivo)
    with gp.Env(empty=True) as env:
        env.setParam("OutputFlag", int(MOSTRAR_LOG))
        env.start()
        resultado = resolver_pli(G, problema, env, gap_max_pct=gap_max_pct)

    melhor = resultado["melhor_viavel"]
    limite = resultado["limite_gurobi"]
    li, ls = (melhor, limite) if problema == "MIS" else (limite, melhor)
    gap_percentual = None
    if li is not None and ls is not None and 0 <= li <= ls:
        if li == ls:
            gap_percentual = 0.0
        elif li == 0:
            gap_percentual = float("inf")
        else:
            gap_percentual = 100 * (ls - li) / li

    return {
        "instancia": arquivo.name.removesuffix(".gz").removesuffix(".txt"),
        "arquivo": str(arquivo),
        "problema": problema,
        "n_vertices": G.number_of_nodes(),
        "n_arestas": G.number_of_edges(),
        "status_gurobi": resultado["status_gurobi"],
        "otimo": resultado["otimo"],
        "limite_inferior": li,
        "limite_superior": ls,
        "gap_percentual": gap_percentual,
        "gap_max_pct": gap_max_pct,
        "atingiu_gap": gap_percentual is not None and gap_percentual <= gap_max_pct,
    }

## Resolver um grafo e um problema

Preencha `GRAFO_INDIVIDUAL` com o nome da instância em `data/raw` ou com o caminho de uma lista de arestas. Escolha `MDS`, `MVC` ou `MIS` em `PROBLEMA_INDIVIDUAL`. O solver busca fechar o intervalo de otimalidade.

In [39]:
GRAFO_INDIVIDUAL =  "roadNet-PA"
PROBLEMA_INDIVIDUAL = "MDS"

resultado_individual = None
if GRAFO_INDIVIDUAL is not None:
    resultado_individual = executar_instancia(GRAFO_INDIVIDUAL, PROBLEMA_INDIVIDUAL)
    display(pd.Series(resultado_individual, dtype=object).to_frame("valor"))

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2883602
Academic license 2883602 - for non-commercial use only - registered to f3___@dac.unicamp.br
Set parameter MIPGap to value 0
Set parameter MIPGapAbs to value 0
Gurobi Optimizer version 14.0.0 build v14.0.0rc1 (linux64 - "Debian GNU/Linux 13 (trixie)")

CPU model: AMD Ryzen 7 6800H with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
MIPGap  0
MIPGapAbs  0

Academic license 2883602 - for non-commercial use only - registered to f3___@dac.unicamp.br
Optimize a model with 1088092 rows, 1088092 columns and 4171888 nonzeros (Min)
Model fingerprint: 0x9354b48d
Model has 1088092 linear objective coefficients
Variable types: 0 continuous, 1088092 integer (1088092 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS ran

,valor
instancia,roadNet-PA
arquivo,/home/fernando/complex-networks/data/raw/roadN...
problema,MDS
n_vertices,1088092
n_arestas,1541898
status_gurobi,INTERRUPTED
otimo,None
limite_inferior,324627.0
limite_superior,330235
gap_percentual,1.727521


## Parar pelo gap percentual entre os limites

Preencha `GRAFO_GAP`, `PROBLEMA_GAP` e `GAP_MAX_PCT`. Use `5` para uma tolerância de 5%. O gap é calculado sobre o limite inferior.

$$
\mathrm{gap}\ (\%) = 100 \cdot \frac{LS - LI}{LI}
$$

Quando os dois limites são zero, o gap é zero. Com $LI=0$ e $LS>0$, o gap é infinito. O notebook converte a tolerância para o parâmetro `MIPGap` do Gurobi e indica em `atingiu_gap` se o intervalo retornado está dentro da tolerância.

Com $LI=100$ e $LS=104$, o gap é 4% e o limite superior fica a no máximo 4% acima do ótimo. Em MDS e MVC, `resultado_gap["limite_superior"]` é a cardinalidade da melhor solução viável. Em MIS, é o limite superior calculado pelo solver.

[Parâmetro MIPGap do Gurobi](https://docs.gurobi.com/projects/optimizer/en/13.0/reference/parameters.html#mipgap)

In [36]:
GRAFO_GAP = "as-skitter"
PROBLEMA_GAP = "MDS"
GAP_MAX_PCT = 1.5  # Use 5 para 5%.

resultado_gap = None
if GRAFO_GAP is not None:
    resultado_gap = executar_instancia(
        GRAFO_GAP,
        PROBLEMA_GAP,
        gap_max_pct=GAP_MAX_PCT,
    )
    display(pd.Series(resultado_gap, dtype=object).to_frame("valor"))

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2883602
Academic license 2883602 - for non-commercial use only - registered to f3___@dac.unicamp.br
Set parameter MIPGap to value 1.4778325123152709e-02
Set parameter MIPGapAbs to value 0
Gurobi Optimizer version 14.0.0 build v14.0.0rc1 (linux64 - "Debian GNU/Linux 13 (trixie)")

CPU model: AMD Ryzen 7 6800H with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
MIPGap  1.4778325123152709e-02
MIPGapAbs  0

Academic license 2883602 - for non-commercial use only - registered to f3___@dac.unicamp.br
Optimize a model with 47538 rows, 47538 columns and 493312 nonzeros (Min)
Model fingerprint: 0x1c76f60b
Model has 47538 linear objective coefficients
Variable types: 0 continuous, 47538 integer (47538 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds ran

,valor
instancia,gemsec-Deezer-hu
arquivo,/home/fernando/complex-networks/data/raw/gemse...
problema,MDS
n_vertices,47538
n_arestas,222887
status_gurobi,OPTIMAL
otimo,None
limite_inferior,7658.0
limite_superior,7772
gap_percentual,1.488639


## Executar em `data/raw`

Defina `EXECUTAR_LOTE = True` na célula abaixo para iniciar o lote. Ele começa pelos menores arquivos em disco e grava o CSV a cada execução. Reexecutar a célula com o lote habilitado inicia um novo lote e sobrescreve esse CSV. Em MDS e MVC, `limite_gurobi` é um limite inferior. Em MIS, é um limite superior.

| Coluna | MDS e MVC (minimização) | MIS (maximização) |
| --- | --- | --- |
| `melhor_viavel` | Limite superior | Limite inferior |
| `limite_gurobi` | Limite inferior | Limite superior |

In [35]:
EXECUTAR_LOTE = True
GAP_MAX_PCT = 1

if EXECUTAR_LOTE:
    arquivos = sorted(
        (p for p in RAW_DATA_DIR.iterdir() if p.is_file() and p.name.endswith((".txt", ".txt.gz"))),
        key=lambda p: (p.stat().st_size, p.name),
    )
    if not arquivos:
        raise FileNotFoundError(f"Nenhuma lista de arestas .txt ou .txt.gz em {RAW_DATA_DIR}.")

    colunas = [
        "instancia", "arquivo", "n_vertices", "n_arestas", "problema", "status_gurobi",
        "otimo", "melhor_viavel", "limite_gurobi", "tempo_ate_gap_s", "detalhe",
    ]
    CSV_RESULTADOS.parent.mkdir(parents=True, exist_ok=True)

    with gp.Env(empty=True) as env:
        env.setParam("OutputFlag", int(MOSTRAR_LOG))
        env.start()     

        with CSV_RESULTADOS.open("w", encoding="utf-8", newline="") as saida:
            writer = csv.DictWriter(saida, fieldnames=colunas)
            writer.writeheader()

            for i, arquivo in enumerate(arquivos, start=1):
                print(f"[{i}/{len(arquivos)}] {arquivo.name}", flush=True)
                base = {
                    "instancia": arquivo.name.removesuffix(".gz").removesuffix(".txt"),
                    "arquivo": arquivo.name,
                }
                try:
                    G = ler_grafo(arquivo)
                except (OSError, EOFError, ValueError) as erro:
                    for problema in PROBLEMAS:
                        writer.writerow(dict(base, problema=problema, status_gurobi="ERRO_LEITURA", detalhe=str(erro)))
                    saida.flush()
                    print(f"    Erro de leitura: {erro}", flush=True)
                    continue

                base.update(n_vertices=G.number_of_nodes(), n_arestas=G.number_of_edges())
                for problema in PROBLEMAS:
                    print(f"    Resolvendo {problema}", flush=True)
                    try:
                        resultado = resolver_pli(G, problema, env, gap_max_pct=GAP_MAX_PCT)
                    except gp.GurobiError as erro:
                        resultado = {"status_gurobi": "ERRO_GUROBI", "detalhe": str(erro)}
                    writer.writerow(dict(base, problema=problema, **resultado))
                    saida.flush()
                    print(f"    {problema} {resultado['status_gurobi']} | ótimo {resultado.get('otimo')}", flush=True)
                    if resultado["status_gurobi"] == "INTERRUPTED":
                        raise KeyboardInterrupt("Execução interrompida no Gurobi.")
                del G
                gc.collect()

    print(f"Resultados em {CSV_RESULTADOS}")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2883602
Academic license 2883602 - for non-commercial use only - registered to f3___@dac.unicamp.br
[1/40] gemsec-Facebook-tvshow.txt.gz
    Resolvendo MDS
Set parameter MIPGap to value 9.9009900990099011e-03
Set parameter MIPGapAbs to value 0
Gurobi Optimizer version 14.0.0 build v14.0.0rc1 (linux64 - "Debian GNU/Linux 13 (trixie)")

CPU model: AMD Ryzen 7 6800H with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
MIPGap  9.9009900990099011e-03
MIPGapAbs  0

Academic license 2883602 - for non-commercial use only - registered to f3___@dac.unicamp.br
Optimize a model with 3892 rows, 3892 columns and 38370 nonzeros (Min)
Model fingerprint: 0x3642ca18
Model has 3892 linear objective coefficients
Variable types: 0 continuous, 3892 integer (3892 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+

KeyboardInterrupt: Execução interrompida no Gurobi.

In [ ]:
if CSV_RESULTADOS.is_file():
    resultados = pd.read_csv(CSV_RESULTADOS, dtype={"otimo": "Int64", "melhor_viavel": "Int64"})
    otimos = resultados.pivot(index="arquivo", columns="problema", values="otimo")
    display(otimos.reindex(columns=PROBLEMAS))

    pendentes = resultados.loc[resultados["otimo"].isna()]
    if not pendentes.empty:
        display(pendentes[["arquivo", "problema", "status_gurobi", "melhor_viavel", "limite_gurobi", "detalhe"]])